# 02 · Incremental — doc-status pipeline + cross-document entity merging

This reads the `lightrag_news` graph built by `ingest.py` (two waves, tagged `wave-1`/`wave-2`) and inspects the ingestion bookkeeping and a recurring, merged entity.

> Run `ingest.py` first.

In [1]:
import sys, pathlib, logging
sys.path.insert(0, str(pathlib.Path.cwd().parent))  # examples/demos
# LightRAG resets its own logger to INFO on construction, so silence verbose
# INFO/WARNING logs globally (logging.disable can't be overridden by setLevel).
logging.disable(logging.WARNING)
from _common import config
from _common.rag import build_rag
from lightrag import QueryParam
from lightrag.kg.shared_storage import initialize_pipeline_status
config.require_openai_key()
try:
    from lightrag.utils import GRAPH_FIELD_SEP
except Exception:
    GRAPH_FIELD_SEP = "<SEP>"
rag = build_rag("lightrag_news")
await rag.initialize_storages(); await initialize_pipeline_status()
g = rag.chunk_entity_relation_graph
print("entities in graph:", f"{len(await g.get_all_labels()):,}")

entities in graph: 2,839


## The ingestion pipeline (doc-status + per-wave `track_id`)

In [2]:
print("status counts:", await rag.doc_status.get_all_status_counts())
for tid in ("wave-1", "wave-2"):
    print(f"  docs tagged {tid}: {len(await rag.doc_status.get_docs_by_track_id(tid))}")

status counts: {'pending': 0, 'parsing': 0, 'analyzing': 0, 'processing': 0, 'preprocessed': 0, 'processed': 273, 'failed': 28, 'all': 301}
  docs tagged wave-1: 150
  docs tagged wave-2: 150


## A recurring entity, merged across documents

In [3]:
hub = (await g.get_popular_labels(limit=1))[0]
node = await g.get_node(hub)
sources = [s for s in (node.get("source_id") or "").split(GRAPH_FIELD_SEP) if s]
print(f"entity: {hub}")
print(f"  degree: {await g.node_degree(hub)}  source chunks: {len(sources)}")
print(f"  description: {(node.get('description') or '')[:160]}")

entity: Charlottesville
  degree: 32  source chunks: 17
  description: Charlottesville is a city in Virginia that gained national attention as a significant site of discourse on race, particularly due to a violent incident related 


## Doc-status pagination (newest first)

In [4]:
rows, total = await rag.doc_status.get_docs_paginated(page=1, page_size=5,
                                                     sort_field="updated_at", sort_direction="desc")
print("total documents:", total)
for doc_id, st in rows:
    print(f"  {doc_id}  status={getattr(st,'status','?')}  {getattr(st,'file_path','?')}")

total documents: 301
  dup-75084e9bd5b0fea623819b514d7a6233  status=failed  mother-daughter-duo-dancing-2516681965.html
  news-208  status=processed  73-year-old-latest-victim-deadly-attacks-mexican-journalists
  news-176  status=processed  4218188-having-ball-seniors-two-step-oldies-parkwood-senior-livings-valentines-social
  news-310  status=processed  prince-harry-talks-about-finally-getting-therapy-1794384403?utm_source=feedburner&utm_medium=feed&utm_campaign=Feed%3A+jezebel%2Ffull+%28Jezebel%29
  news-192  status=processed  best-broadway-comes-kerrville
  news-231  status=processed  nonprofit-working-block-drug-imports-has-ties-pharma-lobby
  news-242  status=processed  2-first-time-boston-marathoners-emerge-victorious
  news-283  status=processed  black-boys-dropping-income-levels-as-adults
  news-266  status=processed  sbi-pnb-ubi-cut-lending-rates-by-up-to-0-9-pc.html
  news-244  status=processed  why-government-cant-bring-terrorism-charges-charlottesville


In [5]:
await rag.finalize_storages()